In [2]:
import sys
import os

from from_root import from_root

sys.path.insert(0, str(from_root("src")))
from read_and_write_docs import read_rds, read_jsonl
from model_loading import load_model
from n_gram_tracing import (
    tokenize_to_tokens,
)
from utils import apply_temp_doc_id, build_metadata_df
from greedy_string_tiling import gst_known_unknown_tiles, print_gst_tiles, show_many_knowns_gst_tiles_html, gst_many_knowns_vs_unknown_tiles

In [3]:
# --- set your args (strings) ---
base_loc = "/Volumes/BCross"

if not os.path.exists(base_loc):
    print("Using Offline Location")
    base_loc = "/Users/user/Documents/uni_work_offline"
else:
    print("Using Volume Location")

CORPUS = "Yelp"
DATA_TYPE = "test"

KNOWN_LOC = f"{base_loc}/datasets/author_verification/{DATA_TYPE}/{CORPUS}/known_raw.jsonl"
UNKNOWN_LOC = f"{base_loc}/datasets/author_verification/{DATA_TYPE}/{CORPUS}/unknown_raw.jsonl"
METADATA_LOC = f"{base_loc}/datasets/author_verification/{DATA_TYPE}/metadata.rds"
MODEL_LOC = f"{base_loc}/models/gpt2"

# file_loc = '/Users/user/Downloads/-as_oxYIDXEGRxLxgd6DGA vs -AXo3Eeg-Ipqx54f8ZahLQ.xlsx'
# PROBLEM = Path(file_loc).stem
PROBLEM = "9q36VyX34T8C9hf2K88afw vs 9qyLpg3JTVUZXS4XrQgZkQ"

print(f"Problem: {PROBLEM}")

Using Volume Location
Problem: 9q36VyX34T8C9hf2K88afw vs 9qyLpg3JTVUZXS4XrQgZkQ


In [4]:
tokenizer, model = load_model(MODEL_LOC)

In [5]:
known = read_jsonl(KNOWN_LOC)
known = apply_temp_doc_id(known)
    
unknown = read_jsonl(UNKNOWN_LOC)
unknown = apply_temp_doc_id(unknown)

metadata = read_rds(METADATA_LOC)
filtered_metadata = metadata[
    (metadata['corpus'] == CORPUS)
    & (metadata['problem'] ==  PROBLEM)
]

agg_metadata = build_metadata_df(filtered_metadata, known, unknown)

# -----
# Get the chosen text & metadata
# -----
    
known_author = filtered_metadata['known_author'].iloc[0]
unknown_author = filtered_metadata['unknown_author'].iloc[0]

selected_known = known[known['author'] == known_author]
selected_unknown = unknown[unknown['author'] == unknown_author]

unknown_doc = selected_unknown['doc_id'].iloc[0]
unknown_text = selected_unknown['text'].iloc[0]

num_known_problems = len(selected_known)
known_texts = selected_known['text'].tolist()
print(f"There are {num_known_problems} known texts in the problem")

There are 5 known texts in the problem


In [6]:
result = gst_known_unknown_tiles(
    known_text=known_texts[0],
    unknown_text=unknown_text,
    tokenizer=tokenizer,
    min_match=1,
    lowercase=True,
    include_token_lists=True
)

In [7]:
result = gst_many_knowns_vs_unknown_tiles(
    known_texts=known_texts,
    unknown_text=unknown_text,
    tokenizer=tokenizer,
    min_match=1,
    lowercase=True,
)

tiles = result["tiles"]
common = result["tile_token_lists"]

In [11]:
common

[['.'],
 [','],
 ['o'],
 ['-'],
 ['Ġ('],
 ['Ġa'],
 ['Ġi'],
 ['Ġ2'],
 [').'],
 ["'s"],
 ['Ġto'],
 ['Ġon'],
 ['Ġof'],
 ['...'],
 ['Ġin'],
 ['Ġme'],
 ['Ġis'],
 ['Ġout'],
 ['Ġand'],
 ['Ġfor'],
 ['Ġthe'],
 ['Ġwas'],
 ['Ġbut'],
 ['Ġyou'],
 ['Ġget'],
 ['Ġwere'],
 ['Ġthat'],
 ['Ġtime'],
 ['Ġname'],
 ['Ġwhen'],
 ['Ġtheir'],
 ['Ġthink'],
 ['Ġright'],
 ['Ġservice'],
 ['Ġnothing'],
 ['.', 'Ġi'],
 [',', 'Ġi'],
 ['.', 'Ġit'],
 ['Ġi', 'Ġam'],
 ['Ġit', "'s"],
 ['Ġat', 'Ġthe'],
 ['.', 'Ġthere'],
 ['Ġis', 'Ġthe'],
 ['Ġi', 'Ġhave'],
 ['Ġand', 'Ġthe'],
 ['Ġbeen', 'Ġto'],
 ['Ġthing', 'Ġthat'],
 ['Ġof', 'Ġall', 'Ġthe']]

In [8]:
len(common)

48

In [10]:
show_many_knowns_gst_tiles_html(result)